# 🧩 Mini-Lab: pytest, Fixtures, and Golden Files

**Module 8: Tests That Catch Real Bugs** | **Type: Mini-Lab (Brick)**

---

## Learning Objectives

By the end of this mini-lab, you will be able to:

1. **Classify** what is deterministic vs stochastic vs integration in an AI app
2. **Write** pytest tests for the cleaner and parser, including a deliberate failure
3. **Reuse** sample data with fixtures, `tmp_path`, and a golden-output comparison

## Target Concepts

| Concept | Gloss |
|---------|-------|
| Deterministic vs stochastic | Test what your code does with output; measure model quality separately |
| Unit test | Small assertion on one behaviour; a test that cannot fail is theatre |
| Test fixture | Reusable sample data, tmp_path, golden-output comparison |


## Why "Fail First" Matters

A test that *can never fail* tells you nothing. Before trusting a green check, you need to see that the test **turns red when the code is wrong**.

This mini-lab walks through a tight loop:

1. Write a pure function (deterministic — recall the testability map from the previous mini-lab)
2. Write a pytest test for it
3. **Break** the function on purpose and confirm the test fails
4. Fix the function and confirm the test passes again

---
## What is actually testable in an AI app


## The Problem: What Should You Actually Test?

An AI application is **not** one big blob. It is a pipeline of stages:

```
User Input → Validate → Build Prompt → Call LLM → Parse Output → Format Response
```

Some of these stages are **deterministic** — same input, same output, every time. Others are **stochastic** — even with the same input, the output can change.

The key insight: **you can write reliable, traditional tests for the deterministic stages** (and there are more of them than you think).

## A Concrete AI Pipeline

Let's look at a simple document-extraction pipeline. The user sends raw text and gets back structured JSON. We'll define each stage as a small function.

In [2]:
import json

# ── Stage 1: Validate Input ──────────────────────────────────
def validate_input(text: str) -> str:
    """Reject empty or too-long inputs."""
    text = text.strip()
    if not text:
        raise ValueError("Input text is empty.")
    if len(text) > 5000:
        raise ValueError("Input text exceeds 5000 characters.")
    return text


# ── Stage 2: Build Prompt ────────────────────────────────────
def build_prompt(text: str) -> list[dict]:
    """Create the messages list for the LLM call."""
    return [
        {"role": "system", "content": "Extract name, email, and company from the text. Reply in JSON."},
        {"role": "user", "content": text},
    ]


# ── Stage 3: Call LLM (simulated) ────────────────────────────
def call_llm(messages: list[dict]) -> str:
    """Simulate an LLM response. In production this hits an API."""
    # Simulated — a real call would be non-deterministic
    return '{"name": "Alice Smith", "email": "alice@example.com", "company": "Acme Inc"}'


# ── Stage 4: Parse Output ────────────────────────────────────
def parse_output(raw: str) -> dict:
    """Parse JSON from the LLM's raw string response."""
    return json.loads(raw)


# ── Stage 5: Format Response ─────────────────────────────────
def format_response(data: dict) -> str:
    """Turn parsed data into a user-friendly string."""
    parts = [f"- **{k.title()}**: {v}" for k, v in data.items()]
    return "\n".join(parts)


print("Pipeline functions defined.")

Pipeline functions defined.


## Mapping Each Stage

Now let's classify every stage. The question for each one is simple:

> **Given the exact same input, will this function *always* return the exact same output?**

If yes → **deterministic** → unit-testable.  
If no → **stochastic** → needs a different evaluation strategy.

In [3]:
pipeline_stages = [
    {
        "stage": "1. Validate Input",
        "function": "validate_input()",
        "behavior": "Deterministic",
        "testable_with_pytest": "✅ Yes",
        "why": "Pure string logic — same input always gives same result or same error.",
    },
    {
        "stage": "2. Build Prompt",
        "function": "build_prompt()",
        "behavior": "Deterministic",
        "testable_with_pytest": "✅ Yes",
        "why": "String formatting — template + input always produces the same messages list.",
    },
    {
        "stage": "3. Call LLM",
        "function": "call_llm()",
        "behavior": "Stochastic",
        "testable_with_pytest": "⚠️ Not directly",
        "why": "Model output varies across runs. Needs evaluation metrics, not assert ==.",
    },
    {
        "stage": "4. Parse Output",
        "function": "parse_output()",
        "behavior": "Deterministic",
        "testable_with_pytest": "✅ Yes",
        "why": "JSON parsing is pure code — same string always yields the same dict (or same error).",
    },
    {
        "stage": "5. Format Response",
        "function": "format_response()",
        "behavior": "Deterministic",
        "testable_with_pytest": "✅ Yes",
        "why": "String formatting — same dict always produces the same output string.",
    },
]

# Build a markdown table
header = "| Stage | Function | Behavior | Testable with pytest? | Why |\n"
header += "|-------|----------|----------|-----------------------|-----|\n"
rows = ""
for s in pipeline_stages:
    rows += f"| {s['stage']} | `{s['function']}` | **{s['behavior']}** | {s['testable_with_pytest']} | {s['why']} |\n"

md(header + rows)

| Stage | Function | Behavior | Testable with pytest? | Why |
|-------|----------|----------|-----------------------|-----|
| 1. Validate Input | `validate_input()` | **Deterministic** | ✅ Yes | Pure string logic — same input always gives same result or same error. |
| 2. Build Prompt | `build_prompt()` | **Deterministic** | ✅ Yes | String formatting — template + input always produces the same messages list. |
| 3. Call LLM | `call_llm()` | **Stochastic** | ⚠️ Not directly | Model output varies across runs. Needs evaluation metrics, not assert ==. |
| 4. Parse Output | `parse_output()` | **Deterministic** | ✅ Yes | JSON parsing is pure code — same string always yields the same dict (or same error). |
| 5. Format Response | `format_response()` | **Deterministic** | ✅ Yes | String formatting — same dict always produces the same output string. |


## Proving It: Deterministic Stages Are Predictable

Let's run the deterministic stages multiple times and confirm the output never changes.

In [4]:
sample_text = "  Contact Alice Smith at alice@example.com, she works at Acme Inc.  "

# Run each deterministic stage 5 times and check consistency
deterministic_stages = {
    "validate_input": lambda: validate_input(sample_text),
    "build_prompt": lambda: build_prompt(sample_text.strip()),
    "parse_output": lambda: parse_output('{"name": "Alice"}'),
    "format_response": lambda: format_response({"name": "Alice", "email": "alice@example.com"}),
}

for name, fn in deterministic_stages.items():
    results = [fn() for _ in range(5)]
    all_same = all(r == results[0] for r in results)
    print(f"{name:20s} → 5 runs identical? {all_same}")

validate_input       → 5 runs identical? True
build_prompt         → 5 runs identical? True
parse_output         → 5 runs identical? True
format_response      → 5 runs identical? True


Every deterministic stage returns the exact same output across all runs. These are the stages you can pin down with simple `assert` statements in pytest.

## The Stochastic Stage: Why `assert ==` Doesn't Work

The LLM call is different. Even with `temperature=0`, outputs can vary slightly across API versions or model updates. Let's simulate this.

In [5]:
import random

def call_llm_realistic(messages: list[dict]) -> str:
    """Simulate slight non-determinism in LLM responses."""
    variations = [
        '{"name": "Alice Smith", "email": "alice@example.com", "company": "Acme Inc"}',
        '{"name": "Alice Smith", "email": "alice@example.com", "company": "Acme Inc."}',
        '{"name":"Alice Smith","email":"alice@example.com","company":"Acme"}',
    ]
    return random.choice(variations)

messages = build_prompt("Contact Alice Smith at alice@example.com, Acme Inc.")
responses = [call_llm_realistic(messages) for _ in range(6)]

print("Six simulated LLM responses:")
for i, r in enumerate(responses, 1):
    print(f"  Run {i}: {r}")

print(f"\nAll identical? {len(set(responses)) == 1}")

Six simulated LLM responses:
  Run 1: {"name": "Alice Smith", "email": "alice@example.com", "company": "Acme Inc."}
  Run 2: {"name": "Alice Smith", "email": "alice@example.com", "company": "Acme Inc"}
  Run 3: {"name": "Alice Smith", "email": "alice@example.com", "company": "Acme Inc"}
  Run 4: {"name": "Alice Smith", "email": "alice@example.com", "company": "Acme Inc."}
  Run 5: {"name": "Alice Smith", "email": "alice@example.com", "company": "Acme Inc."}
  Run 6: {"name":"Alice Smith","email":"alice@example.com","company":"Acme"}

All identical? False


The LLM stage may return slightly different strings each time. You **cannot** write `assert response == expected` and expect it to stay green. That stage requires evaluation strategies (e.g., checking required keys exist, semantic similarity checks) — not exact-match unit tests.

## The Testability Map

Here's the mental model to carry forward:

In [6]:
testability_map = """
## 🗺️ Testability Map

```
┌──────────────────────────────────────────────────────────────┐
│                    AI Pipeline Stages                       │
├──────────────────────────────────────────────────────────────┤
│                                                              │
│  ✅ DETERMINISTIC (unit-testable)                            │
│  ┌────────────┐  ┌─────────────┐  ┌──────────────┐          │
│  │  Validate   │  │ Build Prompt │  │ Parse Output │          │
│  │   Input     │  │  (template)  │  │   (JSON)     │          │
│  └────────────┘  └─────────────┘  └──────────────┘          │
│  ┌─────────────────┐                                         │
│  │ Format Response  │                                         │
│  └─────────────────┘                                         │
│                                                              │
│  ⚠️  STOCHASTIC (needs eval, not assert ==)                  │
│  ┌────────────┐                                              │
│  │  Call LLM   │  ← output varies run to run                │
│  └────────────┘                                              │
│                                                              │
└──────────────────────────────────────────────────────────────┘
```

**Rule of thumb:** Most of your AI app is deterministic glue code.
Test that glue thoroughly, and evaluate the LLM part separately.
"""

md(testability_map)


## 🗺️ Testability Map

```
┌──────────────────────────────────────────────────────────────┐
│                    AI Pipeline Stages                       │
├──────────────────────────────────────────────────────────────┤
│                                                              │
│  ✅ DETERMINISTIC (unit-testable)                            │
│  ┌────────────┐  ┌─────────────┐  ┌──────────────┐          │
│  │  Validate   │  │ Build Prompt │  │ Parse Output │          │
│  │   Input     │  │  (template)  │  │   (JSON)     │          │
│  └────────────┘  └─────────────┘  └──────────────┘          │
│  ┌─────────────────┐                                         │
│  │ Format Response  │                                         │
│  └─────────────────┘                                         │
│                                                              │
│  ⚠️  STOCHASTIC (needs eval, not assert ==)                  │
│  ┌────────────┐                                              │
│  │  Call LLM   │  ← output varies run to run                │
│  └────────────┘                                              │
│                                                              │
└──────────────────────────────────────────────────────────────┘
```

**Rule of thumb:** Most of your AI app is deterministic glue code.
Test that glue thoroughly, and evaluate the LLM part separately.


## Setup

We need `pytest` and the `ipytest` helper that lets us run pytest inside a notebook.

In [1]:
!pip install -q pytest ipytest


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Users\molak\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
import ipytest
ipytest.autoconfig()

print("ipytest ready — we can run pytest tests inside notebook cells.")

ipytest ready — we can run pytest tests inside notebook cells.


## Step 1 — Two Pure Functions to Test

These are the same kind of **deterministic glue-code** functions you identified in the testability map mini-lab. They are *pure*: no network calls, no randomness, no side effects.

In [3]:
import json


def clean_input(text: str) -> str:
    """Strip whitespace and collapse multiple spaces."""
    return " ".join(text.split())


def parse_json_response(raw: str) -> dict:
    """Parse a JSON string; raise ValueError on failure."""
    try:
        return json.loads(raw)
    except json.JSONDecodeError as e:
        raise ValueError(f"Invalid JSON: {e}") from e


print("Pure functions defined.")

Pure functions defined.


### Why are these pure?

- **No randomness** — same input → same output, every time.
- **No I/O or network** — they don't read files, call APIs, or touch a database.
- **No hidden state** — they don't modify globals or rely on mutable class attributes.

Pure functions are the easiest code to test with `assert ==`.

## Step 2 — Write Your First Tests (Green)

A pytest test is just a function whose name starts with `test_` and uses plain `assert` statements. No special classes, no boilerplate.

In [4]:
%%ipytest -v

import pytest


# ── Tests for clean_input ─────────────────────────────────

def test_clean_input_strips_edges():
    assert clean_input("  hello world  ") == "hello world"


def test_clean_input_collapses_spaces():
    assert clean_input("too   many    spaces") == "too many spaces"


# ── Tests for parse_json_response ─────────────────────────

def test_parse_valid_json():
    result = parse_json_response('{"name": "Alice"}')
    assert result == {"name": "Alice"}


def test_parse_invalid_json_raises():
    with pytest.raises(ValueError):
        parse_json_response("not json at all")

======================================= test session starts =======================================
platform win32 -- Python 3.12.10, pytest-9.1.1, pluggy-1.6.0
rootdir: C:\Users\molak\MJ_Projects\19_atomic_ai_eng_course\1_ai_engineering_track\output\content\labs\mini-first-pytest-tests
plugins: anyio-4.9.0, asyncio-1.4.0
asyncio: mode=Mode.STRICT, debug=False, asyncio_default_fixture_loop_scope=None, asyncio_default_test_loop_scope=function
collected 4 items

t_71d3bf8958aa40dda771a336ee1dc93e.py 

.

.

.

.

                                                   [100%]



======================================== 4 passed in 0.08s ========================================


All four tests should show **PASSED**. But how do we know they're actually checking something useful? We need to see them **fail**.

## Step 3 — Break the Code, Watch the Tests Fail (Red)

We'll redefine `clean_input` with a bug — it no longer collapses multiple spaces. If our tests are meaningful, at least one should turn red.

In [5]:
# ── Intentionally broken version ──────────────────────────
def clean_input_broken(text: str) -> str:
    """Bug: only strips edges, does NOT collapse internal spaces."""
    return text.strip()

In [6]:
%%ipytest -v


def test_broken_strips_edges():
    # This still passes — the bug doesn't affect edge stripping
    assert clean_input_broken("  hello world  ") == "hello world"


def test_broken_collapses_spaces():
    # This FAILS — the bug is caught!
    assert clean_input_broken("too   many    spaces") == "too many spaces"

======================================= test session starts =======================================
platform win32 -- Python 3.12.10, pytest-9.1.1, pluggy-1.6.0
rootdir: C:\Users\molak\MJ_Projects\19_atomic_ai_eng_course\1_ai_engineering_track\output\content\labs\mini-first-pytest-tests
plugins: anyio-4.9.0, asyncio-1.4.0
asyncio: mode=Mode.STRICT, debug=False, asyncio_default_fixture_loop_scope=None, asyncio_default_test_loop_scope=function
collected 2 items

t_71d3bf8958aa40dda771a336ee1dc93e.py 

.

F

                                                     [100%]



============================================ FAILURES =============================================
__________________________________ test_broken_collapses_spaces ___________________________________

    def test_broken_collapses_spaces():
        # This FAILS — the bug is caught!
>       assert clean_input_broken("too   many    spaces") == "too many spaces"
E       AssertionError: assert 'too   many    spaces' == 'too many spaces'
E         
E         - too many spaces
E         + too   many    spaces
E         ?    ++     +++

C:\Users\molak\AppData\Local\Temp\ipykernel_60472\1707360044.py:8: AssertionError
===================================== short test summary info =====================================
FAILED t_71d3bf8958aa40dda771a336ee1dc93e.py::test_broken_collapses_spaces - AssertionError: assert 'too   many    spaces' == 'too many spaces'
=================================== 1 failed, 1 passed in 1.23s ===================================


`test_broken_collapses_spaces` should show **FAILED** with a clear diff:

```
assert 'too   many    spaces' == 'too many spaces'
```

**This is the point.** The test caught a real bug. If it hadn't failed, the test would be worthless.

## Step 4 — Fix and Confirm Green Again

Now we run the same assertions against the **correct** function to confirm the cycle: red → fix → green.

In [7]:
%%ipytest -v


def test_fixed_strips_edges():
    assert clean_input("  hello world  ") == "hello world"


def test_fixed_collapses_spaces():
    assert clean_input("too   many    spaces") == "too many spaces"

======================================= test session starts =======================================
platform win32 -- Python 3.12.10, pytest-9.1.1, pluggy-1.6.0
rootdir: C:\Users\molak\MJ_Projects\19_atomic_ai_eng_course\1_ai_engineering_track\output\content\labs\mini-first-pytest-tests
plugins: anyio-4.9.0, asyncio-1.4.0
asyncio: mode=Mode.STRICT, debug=False, asyncio_default_fixture_loop_scope=None, asyncio_default_test_loop_scope=function
collected 2 items

t_71d3bf8958aa40dda771a336ee1dc93e.py 

.

.

                                                     [100%]



======================================== 2 passed in 0.04s ========================================


Both tests are green again. We've completed the cycle:

| Step | What happened |
|------|---------------|
| **Green** | Tests pass against the correct `clean_input` |
| **Red** | Tests fail against `clean_input_broken` — bug detected |
| **Green** | Tests pass again after switching back to the correct function |

A test you've never seen fail is a test you can't trust.

## Quick Reference: Anatomy of a pytest Test

```python
def test_<what_you_are_checking>():        # name starts with test_
    result = my_function(input_value)       # call the code under test
    assert result == expected_value          # plain assert

def test_<error_case>():
    with pytest.raises(SomeException):      # expect a specific error
        my_function(bad_input)
```

That's it. No classes, no decorators, no registration — just functions and `assert`.

---
## Fixtures and a golden file

Make the test data reusable.


## Why Fixtures and Golden Files?

In the previous mini-lab you wrote standalone tests with hard-coded inputs. That works for two or three tests, but as your test suite grows you'll notice:

1. **Repeated setup** — multiple tests need the same sample data, so you copy-paste it everywhere.
2. **Temporary files** — some functions read/write files; you need a safe scratch directory that cleans itself up.
3. **Output drift** — a function's output changes subtly after a refactor; you want to compare against a known-good "golden" snapshot.

Fixtures solve (1) and (2). Golden-file comparisons solve (3).

## Setup

In [1]:
!pip install -q pytest ipytest


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Users\molak\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
import ipytest
ipytest.autoconfig()

print("ipytest ready.")

ipytest ready.


## The Code Under Test

We'll test a small function that takes a list of contact dicts and writes a formatted report to a file. This is exactly the kind of deterministic glue code from the testability map — no LLM involved.

In [3]:
from pathlib import Path


def write_contact_report(contacts: list[dict], output_path: Path) -> Path:
    """Write a formatted contact report to a file and return the path."""
    lines = ["# Contact Report", ""]
    for c in contacts:
        lines.append(f"- {c['name']} ({c['email']})")
    lines.append("")
    lines.append(f"Total: {len(contacts)} contacts")
    output_path.write_text("\n".join(lines))
    return output_path


print("write_contact_report() defined.")

write_contact_report() defined.


## Step 1 — A Reusable Fixture

> 🧩 **Fixtures and test data.** Watch for the same sample contacts to be injected by name into both tests.

A **fixture** is a function decorated with `@pytest.fixture`. Any test that names it as a parameter receives the fixture's return value automatically — no manual calling needed.

```python
@pytest.fixture
def sample_contacts():       # ← define once
    return [{...}, {...}]

def test_something(sample_contacts):   # ← inject by name
    assert len(sample_contacts) == 2
```

This eliminates copy-pasting the same list into every test.


In [4]:
%%ipytest -v

import pytest


@pytest.fixture
def sample_contacts():
    """Reusable test data — defined once, used by many tests."""
    return [
        {"name": "Alice Smith", "email": "alice@example.com"},
        {"name": "Bob Jones", "email": "bob@example.com"},
    ]


# ── Two tests sharing the same fixture ──────────────────────

def test_report_has_all_names(sample_contacts, tmp_path):
    """Every contact name appears in the report."""
    out = tmp_path / "report.md"
    write_contact_report(sample_contacts, out)
    text = out.read_text()
    for contact in sample_contacts:
        assert contact["name"] in text


def test_report_shows_correct_total(sample_contacts, tmp_path):
    """The total line matches the number of contacts."""
    out = tmp_path / "report.md"
    write_contact_report(sample_contacts, out)
    text = out.read_text()
    assert "Total: 2 contacts" in text

======================================= test session starts =======================================
platform win32 -- Python 3.12.10, pytest-9.1.1, pluggy-1.6.0
rootdir: C:\Users\molak\MJ_Projects\19_atomic_ai_eng_course\1_ai_engineering_track\output\content\labs\mini-fixtures-and-golden-file
plugins: anyio-4.9.0, asyncio-1.4.0
asyncio: mode=Mode.STRICT, debug=False, asyncio_default_fixture_loop_scope=None, asyncio_default_test_loop_scope=function
collected 2 items

t_ed232d2137394a1ab4e522ea08db2dc7.py 

.

.

                                                     [100%]


======================================== 2 passed in 0.25s ========================================


### What just happened?

- `sample_contacts` — our custom fixture, injected by name into both tests.
- `tmp_path` — a **built-in pytest fixture** that gives each test its own temporary directory as a `pathlib.Path`. It is automatically cleaned up after the test finishes.

Neither test had to create or delete temporary files manually. Fixtures handled everything.

## Step 2 — `tmp_path` in Action

`tmp_path` is cross-platform (works on Mac, Windows, Linux) and gives each test a **unique** directory, so tests never interfere with each other.

Let's prove the isolation:

In [5]:
%%ipytest -v

collected_paths = []


def test_tmp_path_a(tmp_path):
    """Write a file in this test's temp dir."""
    f = tmp_path / "a.txt"
    f.write_text("from test a")
    collected_paths.append(tmp_path)
    assert f.exists()


def test_tmp_path_b(tmp_path):
    """This test gets a completely different temp dir."""
    f = tmp_path / "a.txt"          # same filename…
    assert not f.exists()            # …but it doesn't exist here
    collected_paths.append(tmp_path)


def test_paths_are_unique():
    """Confirm the two tests got different directories."""
    assert len(collected_paths) == 2
    assert collected_paths[0] != collected_paths[1]

======================================= test session starts =======================================
platform win32 -- Python 3.12.10, pytest-9.1.1, pluggy-1.6.0
rootdir: C:\Users\molak\MJ_Projects\19_atomic_ai_eng_course\1_ai_engineering_track\output\content\labs\mini-fixtures-and-golden-file
plugins: anyio-4.9.0, asyncio-1.4.0
asyncio: mode=Mode.STRICT, debug=False, asyncio_default_fixture_loop_scope=None, asyncio_default_test_loop_scope=function
collected 3 items

t_ed232d2137394a1ab4e522ea08db2dc7.py 

.

.

.

                                                    [100%]



======================================== 3 passed in 0.13s ========================================


Each test got its own sandbox. File `a.txt` written in test A doesn't leak into test B. This is why `tmp_path` is the standard approach for testing file I/O.

## Step 3 — Golden-File Comparison

A **golden file** is a saved copy of the "known-good" output. Your test generates fresh output and compares it character-by-character against the golden file. If anything changes — even a missing newline — the test fails.

This is how you catch **output drift**: subtle changes introduced by refactoring that no individual `assert` would catch.

### The workflow

1. Run the function and save its output as the golden file (done once).
2. In the test, run the function again and compare against the golden file.
3. If the output changes intentionally, update the golden file.

In [6]:
%%ipytest -v

import pytest


# ── Our known-good "golden" output ────────────────────────
GOLDEN_REPORT = """# Contact Report

- Alice Smith (alice@example.com)
- Bob Jones (bob@example.com)

Total: 2 contacts"""


@pytest.fixture
def sample_contacts():
    return [
        {"name": "Alice Smith", "email": "alice@example.com"},
        {"name": "Bob Jones", "email": "bob@example.com"},
    ]


@pytest.fixture
def golden_path(tmp_path):
    """Write the golden file into the test's temp dir."""
    gf = tmp_path / "golden_report.md"
    gf.write_text(GOLDEN_REPORT)
    return gf


def test_output_matches_golden(sample_contacts, golden_path, tmp_path):
    """Generate a fresh report and compare it to the golden file."""
    actual_path = tmp_path / "actual_report.md"
    write_contact_report(sample_contacts, actual_path)

    actual = actual_path.read_text()
    expected = golden_path.read_text()

    assert actual == expected, (
        f"Output drift detected!\n"
        f"--- Expected ---\n{expected}\n"
        f"--- Actual ---\n{actual}"
    )

======================================= test session starts =======================================
platform win32 -- Python 3.12.10, pytest-9.1.1, pluggy-1.6.0
rootdir: C:\Users\molak\MJ_Projects\19_atomic_ai_eng_course\1_ai_engineering_track\output\content\labs\mini-fixtures-and-golden-file
plugins: anyio-4.9.0, asyncio-1.4.0
asyncio: mode=Mode.STRICT, debug=False, asyncio_default_fixture_loop_scope=None, asyncio_default_test_loop_scope=function
collected 1 item

t_ed232d2137394a1ab4e522ea08db2dc7.py 

.

                                                      [100%]



======================================== 1 passed in 0.13s ========================================


The golden-file test passed — the function's output matches the known-good snapshot exactly.

Notice how fixtures compose: `golden_path` itself uses `tmp_path`. Pytest wires them together automatically.

## Step 4 — Break the Code, Watch the Golden Test Fail

As with the previous mini-lab: a test you've never seen fail is a test you can't trust. Let's introduce a subtle bug and confirm the golden comparison catches it.

In [7]:
def write_contact_report_buggy(contacts: list[dict], output_path: Path) -> Path:
    """Bug: uses semicolon instead of comma in the parenthetical."""
    lines = ["# Contact Report", ""]
    for c in contacts:
        lines.append(f"- {c['name']} ({c['email']})")
    lines.append("")
    lines.append(f"Total: {len(contacts)} contact(s)")  # ← subtle change
    output_path.write_text("\n".join(lines))
    return output_path


print("Buggy version defined — 'contacts' → 'contact(s)' in the total line.")

Buggy version defined — 'contacts' → 'contact(s)' in the total line.


In [8]:
%%ipytest -v


@pytest.fixture
def sample_contacts():
    return [
        {"name": "Alice Smith", "email": "alice@example.com"},
        {"name": "Bob Jones", "email": "bob@example.com"},
    ]


@pytest.fixture
def golden_path(tmp_path):
    gf = tmp_path / "golden_report.md"
    gf.write_text(GOLDEN_REPORT)
    return gf


def test_buggy_output_vs_golden(sample_contacts, golden_path, tmp_path):
    """This SHOULD FAIL — the buggy function produces different output."""
    actual_path = tmp_path / "actual_report.md"
    write_contact_report_buggy(sample_contacts, actual_path)

    actual = actual_path.read_text()
    expected = golden_path.read_text()

    assert actual == expected, (
        f"Output drift detected!\n"
        f"--- Expected ---\n{expected}\n"
        f"--- Actual ---\n{actual}"
    )

======================================= test session starts =======================================
platform win32 -- Python 3.12.10, pytest-9.1.1, pluggy-1.6.0
rootdir: C:\Users\molak\MJ_Projects\19_atomic_ai_eng_course\1_ai_engineering_track\output\content\labs\mini-fixtures-and-golden-file
plugins: anyio-4.9.0, asyncio-1.4.0
asyncio: mode=Mode.STRICT, debug=False, asyncio_default_fixture_loop_scope=None, asyncio_default_test_loop_scope=function
collected 1 item

t_ed232d2137394a1ab4e522ea08db2dc7.py 

F

                                                      [100%]



============================================ FAILURES =============================================
___________________________________ test_buggy_output_vs_golden ___________________________________

sample_contacts = [{'name': 'Alice Smith', 'email': 'alice@example.com'}, {'name': 'Bob Jones', 'email': 'bob@example.com'}]
golden_path = WindowsPath('C:/Users/molak/AppData/Local/Temp/pytest-of-molak/pytest-148/test_buggy_output_vs_golden0/golden_report.md')
tmp_path = WindowsPath('C:/Users/molak/AppData/Local/Temp/pytest-of-molak/pytest-148/test_buggy_output_vs_golden0')

    def test_buggy_output_vs_golden(sample_contacts, golden_path, tmp_path):
        """This SHOULD FAIL — the buggy function produces different output."""
        actual_path = tmp_path / "actual_report.md"
        write_contact_report_buggy(sample_contacts, actual_path)
    
        actual = actual_path.read_text()
        expected = golden_path.read_text()
    
>       assert actual == expected, (
            f"O

The test failed and the diff shows exactly what changed: `contacts` → `contact(s)`. That's the power of golden-file testing — it catches any output drift, no matter how small.

## Quick Reference

| Tool | What it does | When to use it |
|------|-------------|----------------|
| `@pytest.fixture` | Supplies reusable data/resources to tests by name | Whenever 2+ tests need the same setup |
| `tmp_path` | Built-in fixture; gives each test a unique temp directory (`pathlib.Path`) | Any test that reads or writes files |
| Golden-file comparison | Compares current output against a saved known-good snapshot | Catching subtle output drift after refactors |

## 🎯 Summary

### Key Takeaways

1. **A pytest test is just a function** — name it `test_*`, use plain `assert`, and pytest finds and runs it automatically
2. **Pure functions are ideal test targets** — no randomness, no I/O, no hidden state means `assert ==` is reliable
3. **Every test must be able to fail** — if you've never seen a test go red, you don't know it's actually checking anything
4. **Red → Green is the discipline** — break the code on purpose, see the failure, fix it, and confirm green before moving on